# Hospital Inpatient Charges Prediction

Modelling notebook: picks up where `hospital_charges_eda.ipynb` left off. Reads the
cleaned data that notebook wrote to `data/processed/cleanedInpatientCharges.csv` and covers
encoding, train/test split, OLS/Ridge/Lasso fitting, evaluation, and the final write-up

## Load Processed Data

Read the already-cleaned, already-trimmed DataFrame produced by the EDA notebook — no need
to repeat the raw load/clean/drop steps here.

In [1]:
import pandas as pd

from insurance_charges_prediction.config import CLEANED_DATA_FILE
from insurance_charges_prediction.features import encode_categoricals
from insurance_charges_prediction.features import build_X_y

df = pd.read_csv(CLEANED_DATA_FILE)
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 163065 entries, 0 to 163064
Data columns (total 8 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   DRG Definition                        163065 non-null  str    
 1   Provider City                         163065 non-null  str    
 2   Provider State                        163065 non-null  str    
 3   Hospital Referral Region Description  163065 non-null  str    
 4   Total Discharges                      163065 non-null  int64  
 5   Average Covered Charges               163065 non-null  float64
 6   Average Total Payments                163065 non-null  float64
 7   Average Medicare Payments             163065 non-null  float64
dtypes: float64(3), int64(1), str(4)
memory usage: 10.0 MB


,DRG Definition,Provider City,Provider State,Hospital Referral Region Description,Total Discharges,Average Covered Charges,Average Total Payments,Average Medicare Payments
0,039 - EXTRACRANIAL PROCEDURES W/O CC/MCC,DOTHAN,AL,AL - Dothan,91,32963.07,5777.24,4763.73
1,039 - EXTRACRANIAL PROCEDURES W/O CC/MCC,BOAZ,AL,AL - Birmingham,14,15131.85,5787.57,4976.71
2,039 - EXTRACRANIAL PROCEDURES W/O CC/MCC,FLORENCE,AL,AL - Birmingham,24,37560.37,5434.95,4453.79
3,039 - EXTRACRANIAL PROCEDURES W/O CC/MCC,BIRMINGHAM,AL,AL - Birmingham,25,13998.28,5417.56,4129.16
4,039 - EXTRACRANIAL PROCEDURES W/O CC/MCC,ALABASTER,AL,AL - Birmingham,18,31633.27,5658.33,4851.44


## One-hot Encoding


In [2]:
df["DRG Definition"].value_counts()

DRG Definition
194 - SIMPLE PNEUMONIA & PLEURISY W CC                                      3023
690 - KIDNEY & URINARY TRACT INFECTIONS W/O MCC                             2989
292 - HEART FAILURE & SHOCK W CC                                            2953
392 - ESOPHAGITIS, GASTROENT & MISC DIGEST DISORDERS W/O MCC                2950
641 - MISC DISORDERS OF NUTRITION,METABOLISM,FLUIDS/ELECTROLYTES W/O MCC    2899
                                                                            ... 
315 - OTHER CIRCULATORY SYSTEM DIAGNOSES W CC                                859
473 - CERVICAL SPINAL FUSION W/O CC/MCC                                      846
917 - POISONING & TOXIC EFFECTS OF DRUGS W MCC                               843
251 - PERC CARDIOVASC PROC W/O CORONARY ARTERY STENT W/O MCC                 727
885 - PSYCHOSES                                                              613
Name: count, Length: 100, dtype: int64

In [3]:
df["DRG Definition"].value_counts().min()

np.int64(613)

In [4]:
encoded_df = encode_categoricals(df)

### Sanity Check

In [5]:
encoded_df.shape

(163065, 149)

In [6]:
encoded_df.head()

,Provider State_AL,Provider State_AR,Provider State_AZ,Provider State_CA,Provider State_CO,Provider State_CT,Provider State_DC,Provider State_DE,Provider State_FL,Provider State_GA,...,DRG Definition_812 - RED BLOOD CELL DISORDERS W/O MCC,DRG Definition_853 - INFECTIOUS & PARASITIC DISEASES W O.R. PROCEDURE W MCC,DRG Definition_870 - SEPTICEMIA OR SEVERE SEPSIS W MV 96+ HOURS,DRG Definition_871 - SEPTICEMIA OR SEVERE SEPSIS W/O MV 96+ HOURS W MCC,DRG Definition_872 - SEPTICEMIA OR SEVERE SEPSIS W/O MV 96+ HOURS W/O MCC,DRG Definition_885 - PSYCHOSES,DRG Definition_897 - ALCOHOL/DRUG ABUSE OR DEPENDENCE W/O REHABILITATION THERAPY W/O MCC,DRG Definition_917 - POISONING & TOXIC EFFECTS OF DRUGS W MCC,DRG Definition_918 - POISONING & TOXIC EFFECTS OF DRUGS W/O MCC,DRG Definition_948 - SIGNS & SYMPTOMS W/O MCC
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


`drop="first"` behaved as expected across all columns

### Encoding Choice
**Provider State** and **DRG Definition** are categorical variables, one-hot encoding was used for both variables

For **Provider State**, direct one-hot encoding made the most sense because there are relatively few states and each category represents a meaningful characteristic that may be associated with differences in hospital charges.

For **DRG Definition** one-hot encoding was also used instead of bucketing. This is because, although there are 100 unique DRG Definitions, the one that appears least frequently appears 613 times, which is still a lot. Therefore, there is no need to combine the less frequent DRG definitions into to obtain more stable estimates.Keeping DRG definitions separate also preserves information about specific inpatient procedures or conditions associated with each observation.

## Set the Dependent and Independent Variables for the Model

In [7]:
X, y = build_X_y(df, encoded_df)